## 1. Imports

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

import joblib
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\A I TECH\OneDrive\Desktop\Project\SentiCare-Capstone")
print("Project Root:", PROJECT_ROOT)

Project Root: C:\Users\A I TECH\OneDrive\Desktop\Project\SentiCare-Capstone


## 2. Load Data

In [7]:
df = pd.read_csv(PROJECT_ROOT / "data/raw/Depression.csv")
print("Raw Shape:", df.shape)
df.head()

Raw Shape: (2028, 18)


,1. Age,2. Gender,3. University,4. Department,5. Academic Year,6. Current CGPA,7. Did you receive a waiver or scholarship at your university?,"1. In a semester, how often have you had little interest or pleasure in doing things?","2. In a semester, how often have you been feeling down, depressed or hopeless?","3. In a semester, how often have you had trouble falling or staying asleep, or sleeping too much?","4. In a semester, how often have you been feeling tired or having little energy?","5. In a semester, how often have you had poor appetite or overeating?","6. In a semester, how often have you been feeling bad about yourself - or that you are a failure or have let yourself or your family down?","7. In a semester, how often have you been having trouble concentrating on things, such as reading the books or watching television?","8. In a semester, how often have you moved or spoke too slowly for other people to notice? Or you've been moving a lot more than usual because you've been restless?","9. In a semester, how often have you had thoughts that you would be better off dead, or of hurting yourself?",Depression Value,Depression Label
0,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,2,2,3,2,2,2,2,3,2,20,Severe Depression
1,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,2,2,2,2,2,2,2,2,19,Moderately Severe Depression
2,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,0,0,0,0,0,0,0,0,0,0,No Depression
3,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,2,1,2,1,2,1,2,2,1,14,Moderate Depression
4,18-22,Male,North South University (NSU),Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,1,3,3,3,1,3,0,3,3,20,Severe Depression


## 3. Exploratory Data Analysis

In [8]:
print(" Null Values ")
print(df.isnull().sum())

print("Target Distribution ")
print(df["Depression Label"].value_counts())

print(" Age ")
print(df["1. Age"].value_counts())

print("CGPA ")
print(df["6. Current CGPA"].value_counts())

print("Academic Year ")
print(df["5. Academic Year"].value_counts())

print("Gender")
print(df["2. Gender"].value_counts())

print(" Scholarship ")
print(df["7. Did you receive a waiver or scholarship at your university?"].value_counts())

 Null Values 
1. Age                                                                                                                                                                   0
2. Gender                                                                                                                                                                0
3. University                                                                                                                                                            0
4. Department                                                                                                                                                            0
5. Academic Year                                                                                                                                                         0
6. Current CGPA                                                                                                                    

## 4. Drop Columns



In [9]:
df = df.drop(columns=[
    "3. University",
    "4. Department",
    "Depression Value"   # CRITICAL: drop before split — direct leakage
])

print("Shape after dropping:", df.shape)

Shape after dropping: (2028, 15)


## 5. Clean Ambiguous Category Values

 in CGPA and Academic Year has no ordinal meaning — replace with NaN so imputer handles it correctly.

In [10]:
df["6. Current CGPA"] = df["6. Current CGPA"].replace("Other", np.nan)
df["5. Academic Year"] = df["5. Academic Year"].replace("Other", np.nan)

print("CGPA nulls:", df["6. Current CGPA"].isnull().sum())
print("Academic Year nulls:", df["5. Academic Year"].isnull().sum())

CGPA nulls: 171
Academic Year nulls: 72


## 6. Map Target to 3 Clinical Levels



This mapping is clinically grounded:
- **low** = PHQ-9 0–4 (minimal/no depression)
- **medium** = PHQ-9 5–14 (mild/moderate depression)
- **high** = PHQ-9 15–27 (moderately severe/severe depression)

In [11]:
def map_depression_label(label):
    if label in ["No Depression", "Minimal Depression"]:
        return "low"
    elif label in ["Mild Depression", "Moderate Depression"]:
        return "medium"
    else:  # Moderately Severe, Severe
        return "high"

df["Depression Label"] = df["Depression Label"].apply(map_depression_label)

print("Mapped target distribution:")
print(df["Depression Label"].value_counts())

Mapped target distribution:
Depression Label
high      1016
medium     871
low        141
Name: count, dtype: int64


## 7. Detect PHQ-9 Columns

In [12]:
PHQ_COLS = [
    col for col in df.columns
    if (
        "doing things"      in col.lower() or
        "depressed"         in col.lower() or
        "sleep"             in col.lower() or
        "tired"             in col.lower() or
        "appetite"          in col.lower() or
        "failure"           in col.lower() or
        "concentrating"     in col.lower() or
        "restless"          in col.lower() or
        "hurting yourself"  in col.lower()
    )
]

print(f"PHQ-9 columns detected: {len(PHQ_COLS)}")
for c in PHQ_COLS:
    print(" -", c[:70])

PHQ-9 columns detected: 9
 - 1. In a semester, how often have you had little interest or pleasure i
 - 2. In a semester, how often have you been feeling down, depressed or h
 - 3. In a semester, how often have you had trouble falling or staying as
 - 4. In a semester, how often have you been feeling tired or having litt
 - 5. In a semester, how often have you had poor appetite or overeating? 
 - 6. In a semester, how often have you been feeling bad about yourself -
 - 7. In a semester, how often have you been having trouble concentrating
 - 8. In a semester, how often have you moved or spoke too slowly for oth
 - 9. In a semester, how often have you had thoughts that you would be be


## 8. Train / Validation Split



In [13]:
TARGET_COL = "Depression Label"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y          
)

# Fit LabelEncoder ONLY on train
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc   = label_encoder.transform(y_val)    # transform only

print("Label classes:", label_encoder.classes_)
print("  0 =", label_encoder.classes_[0])
print("  1 =", label_encoder.classes_[1])
print("  2 =", label_encoder.classes_[2])
print()
print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print()
print("Train class distribution:")
print(y_train.value_counts())
print()
print("Val class distribution:")
print(y_val.value_counts())

Label classes: ['high' 'low' 'medium']
  0 = high
  1 = low
  2 = medium

Train: (1622, 14)
Val:   (406, 14)

Train class distribution:
Depression Label
high      812
medium    697
low       113
Name: count, dtype: int64

Val class distribution:
Depression Label
high      204
medium    174
low        28
Name: count, dtype: int64


## 9. PHQ Feature Engineer



In [14]:
class PHQFeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, phq_cols):
        self.phq_cols = phq_cols

    def fit(self, X, y=None):
        return self   # stateless 

    def transform(self, X):
        X = X.copy()

        
        X["severe_symptom_count"] = (X[self.phq_cols] >= 2).sum(axis=1)

       
        X["any_symptom_count"] = (X[self.phq_cols] >= 1).sum(axis=1)

        
        X = X.drop(columns=self.phq_cols)

        return X

## 10. Define Column Types for Preprocessor

After PHQFeatureEngineer drops raw PHQ cols, the remaining numerical columns are only the non-PHQ ones plus the two engineered features.

In [15]:
ENGINEERED_COLS = ["severe_symptom_count", "any_symptom_count"]

# Non-PHQ numerical columns 
NON_PHQ_NUMERICAL = [
    col for col in X_train.columns
    if X_train[col].dtype in ["int64", "float64"]
    and col not in PHQ_COLS
]

NUMERICAL_COLS = NON_PHQ_NUMERICAL + ENGINEERED_COLS

# Ordinal
ORDINAL_COLS = ["1. Age", "5. Academic Year", "6. Current CGPA"]

AGE_ORDER = ["Below 18", "18-22", "23-26", "27-30", "Above 30"]

YEAR_ORDER = [
    "First Year or Equivalent",
    "Second Year or Equivalent",
    "Third Year or Equivalent",
    "Fourth Year or Equivalent"
]

CGPA_ORDER = [
    "Below 2.50", "2.50 - 2.99", "3.00 - 3.39",
    "3.40 - 3.79", "3.80 - 4.00"
]

ORDINAL_ORDER = [AGE_ORDER, YEAR_ORDER, CGPA_ORDER]

# Nominal
NOMINAL_COLS = [
    "2. Gender",
    "7. Did you receive a waiver or scholarship at your university?"
]

print("Ordinal cols :", ORDINAL_COLS)
print("Nominal cols :", NOMINAL_COLS)
print("Numerical cols:", NUMERICAL_COLS)

Ordinal cols : ['1. Age', '5. Academic Year', '6. Current CGPA']
Nominal cols : ['2. Gender', '7. Did you receive a waiver or scholarship at your university?']
Numerical cols: ['severe_symptom_count', 'any_symptom_count']


## 11. Build Preprocessing Pipelines

In [16]:
# OrdinalEncoder
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=ORDINAL_ORDER,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# OneHotEncoder 
nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# StandardScaler 
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal",   ordinal_pipeline,   ORDINAL_COLS),
        ("nominal",   nominal_pipeline,   NOMINAL_COLS),
        ("numerical", numerical_pipeline, NUMERICAL_COLS),
    ],
    remainder="drop"   
)

## 12. Define Model

In [17]:
model = RandomForestClassifier(
    n_estimators=300,       
    max_depth=10,           
    min_samples_leaf=2,     
    max_features="sqrt",    
    class_weight="balanced",
    random_state=42
)

## 13. Full Pipeline

 PHQ Engineer → Preprocessor → Model



In [18]:
full_pipeline = Pipeline([
    ("phq_engineer", PHQFeatureEngineer(phq_cols=PHQ_COLS)),
    ("preprocessor", preprocessor),
    ("model",        model)
])

full_pipeline.fit(X_train, y_train_enc)
print("Model training completed.")

Model training completed.


## 14. Cross-Validation (on train only)

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    full_pipeline,
    X_train,
    y_train_enc,
    cv=cv,
    scoring="f1_macro"
)

print("── Cross Validation Results (5-fold on train set) ──")
print("Fold Scores:", cv_scores.round(4))
print("Mean Accuracy:", round(cv_scores.mean(), 4))
print("Std:          ", round(cv_scores.std(), 4))

── Cross Validation Results (5-fold on train set) ──
Fold Scores: [0.945  0.9392 0.9295 0.9409 0.9537]
Mean Accuracy: 0.9417
Std:           0.0079


## 15. Validation Set Evaluation

In [20]:
val_preds = full_pipeline.predict(X_val)

print("── Validation Accuracy ──")
print(round(accuracy_score(y_val_enc, val_preds), 4))

print("── Weighted F1 Score ──")
print(round(f1_score(y_val_enc, val_preds, average="weighted"), 4))

print("── Classification Report ──")
print(classification_report(
    y_val_enc,
    val_preds,
    target_names=label_encoder.classes_
))

print("── Confusion Matrix ──")
print(confusion_matrix(y_val_enc, val_preds))

── Validation Accuracy ──
0.9212
── Weighted F1 Score ──
0.9211
── Classification Report ──
              precision    recall  f1-score   support

        high       0.93      0.92      0.92       204
         low       0.97      1.00      0.98        28
      medium       0.90      0.91      0.91       174

    accuracy                           0.92       406
   macro avg       0.93      0.94      0.94       406
weighted avg       0.92      0.92      0.92       406

── Confusion Matrix ──
[[187   0  17]
 [  0  28   0]
 [ 14   1 159]]


## 16. Feature Importance

In [21]:
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances   = full_pipeline.named_steps["model"].feature_importances_

importance_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print("── Top 15 Feature Importances ──")
print(importance_df.head(15).round(4))

── Top 15 Feature Importances ──
numerical__severe_symptom_count                                                0.4796
numerical__any_symptom_count                                                   0.4666
ordinal__6. Current CGPA                                                       0.0159
ordinal__5. Academic Year                                                      0.0153
ordinal__1. Age                                                                0.0077
nominal__2. Gender_Male                                                        0.0046
nominal__7. Did you receive a waiver or scholarship at your university?_Yes    0.0035
nominal__2. Gender_Female                                                      0.0034
nominal__7. Did you receive a waiver or scholarship at your university?_No     0.0033
nominal__2. Gender_Prefer not to say                                           0.0001
dtype: float64


## 17. Save Artifact

In [22]:
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/depression_classification"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "pipeline":        full_pipeline,
        "label_encoder":   label_encoder,
        "feature_columns": X_train.columns.tolist(),
        "phq_columns":     PHQ_COLS
    },
    ARTIFACT_DIR / "depression_bundle.joblib"
)

print("Saved to:", ARTIFACT_DIR / "depression_bundle.joblib")
print("Bundle keys: pipeline, label_encoder, feature_columns, phq_columns")

Saved to: C:\Users\A I TECH\OneDrive\Desktop\Project\SentiCare-Capstone\artifacts\depression_classification\depression_bundle.joblib
Bundle keys: pipeline, label_encoder, feature_columns, phq_columns


## 18. Verify Saved Model

In [23]:
loaded = joblib.load(ARTIFACT_DIR / "depression_bundle.joblib")

sample = X_val.iloc[:1]
pred   = loaded["pipeline"].predict(sample)
label  = loaded["label_encoder"].inverse_transform(pred)

print("Sample input:")
print(sample.to_dict(orient="records")[0])
print()
print("Predicted label:", label[0])
print("Model verified correctly.")

Sample input:
{'1. Age': '23-26', '2. Gender': 'Male', '5. Academic Year': 'Fourth Year or Equivalent', '6. Current CGPA': '3.00 - 3.39', '7. Did you receive a waiver or scholarship at your university?': 'No', '1. In a semester, how often have you had little interest or pleasure in doing things?': 2, '2. In a semester, how often have you been feeling down, depressed or hopeless?': 1, '3. In a semester, how often have you had trouble falling or staying asleep, or sleeping too much? ': 1, '4. In a semester, how often have you been feeling tired or having little energy? ': 1, '5. In a semester, how often have you had poor appetite or overeating? ': 1, '6. In a semester, how often have you been feeling bad about yourself - or that you are a failure or have let yourself or your family down? ': 2, '7. In a semester, how often have you been having trouble concentrating on things, such as reading the books or watching television? ': 0, "8. In a semester, how often have you moved or spoke too s